# Agentic RAG with Local LLM and Embeddings

This notebook demonstrates the implementation of an **Agentic Retrieval-Augmented Generation (RAG)** system using:
- **Local LLM Model**: Qwen2.5 from Hugging Face
- **Local Embedding Model**: all-MiniLM-L6-v2 from Hugging Face
- **Agentic Framework**: smolagents with CodeAgent for reasoning and tool use
- **Multiple Document Types**: PDF, Text, and HTML files
- **Vector Store**: ChromaDB for efficient document retrieval
- **Evaluation Agents**: Specialized agents for RAG quality assessment

## What Makes This Agentic?

Traditional RAG systems follow a fixed pipeline: retrieve → format → generate. **Agentic RAG** empowers the system with:

1. **Reasoning Capabilities**: The agent can think through problems step-by-step
2. **Tool Usage**: The agent decides when and how to use retrieval tools
3. **Adaptive Behavior**: The agent can retry, refine queries, and self-correct
4. **Transparency**: Verbose mode shows the agent's reasoning process
5. **Multi-Step Planning**: The agent can break down complex queries

The **smolagents** library with **CodeAgent** enables these capabilities by allowing the agent to write and execute Python code to accomplish tasks, making decisions about tool usage dynamically.

## Business Application and Context

Imagine having an intelligent assistant that doesn't just retrieve documents—it **reasons** about your question, determines the best approach, and iteratively refines its search strategy. This agentic approach:

- **Adaptive Retrieval**: The agent can reformulate queries if initial results are insufficient
- **Multi-Step Reasoning**: Complex questions are broken down into logical steps
- **Self-Evaluation**: The agent can assess answer quality and retrieve more context if needed
- **Debugging Transparency**: Verbose mode lets you understand the agent's decision-making process

This approach is particularly valuable for:
- Technical documentation Q&A systems
- Research paper analysis
- Complex troubleshooting scenarios
- Multi-document synthesis tasks

## Agentic RAG vs Traditional RAG

| **Aspect** | **Traditional RAG** | **Agentic RAG** |
|------------|-------------------|----------------|
| **Query Processing** | Fixed: Embed → Retrieve → Generate | Dynamic: Agent decides strategy |
| **Retrieval Strategy** | Single retrieval step | Can retry with refined queries |
| **Reasoning** | Limited to prompt engineering | Explicit step-by-step reasoning |
| **Tool Use** | Hardcoded pipeline | Agent decides when to use tools |
| **Adaptability** | Static workflow | Can adapt based on results |
| **Transparency** | Black box generation | Visible reasoning with verbose mode |
| **Error Handling** | No self-correction | Can detect and fix issues |

### Key Advantage: CodeAgent

The **CodeAgent** from smolagents allows the LLM to write and execute Python code, enabling:
- Dynamic tool invocation
- Intermediate result inspection
- Complex multi-step workflows
- Built-in debugging capabilities

## 0 - Prerequisites and Installation

Before running this notebook, ensure you have sufficient system resources:
- **RAM**: At least 8GB (16GB recommended for larger models)
- **Storage**: At least 10GB free space for models
- **GPU**: Optional but recommended for faster inference (CUDA-compatible GPU)

Install the required packages below.

# Install required packages

## Install PyTorch with CUDA support (if you have a compatible GPU)

```shell
pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
```

## Install other dependencies
```shell
pip install -q transformers \
              sentence-transformers \
              langchain \
              langchain-community \
              langchain-text-splitters \
              langchain-huggingface \
              chromadb \
              pypdf \
              beautifulsoup4 \
              lxml \
              accelerate \
              bitsandbytes \
              unstructured \
              smolagents
```

**Note:** The `smolagents` library is essential for building agentic systems with tool use and reasoning capabilities.

**Note:** After installation, you may need to restart the runtime. If you encounter any import errors, restart the kernel and continue from the next cell.

## 1 - Import Required Libraries

Import all necessary libraries for document processing, embeddings, vector storage, LLM interaction, and agentic capabilities.

In [1]:
# Standard library imports
import json
import os
import warnings
import glob
from pathlib import Path

# Data processing imports
import numpy as np
import pandas as pd

# LangChain imports for document loading and processing
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredHTMLLoader,
    DirectoryLoader
)
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Hugging Face Transformers imports for local LLM
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
import torch

# Smolagents imports for agentic capabilities
from smolagents import (
    CodeAgent,
    Tool,
    #HfApiModel
)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")
print("✓ Smolagents library loaded - ready for agentic RAG!")

c:\venv\AgenticAICertv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All libraries imported successfully!
✓ Smolagents library loaded - ready for agentic RAG!


## 2 - Load Local LLM Model (Qwen2.5)

We will load the Qwen2.5 model from Hugging Face. This is a powerful open-source language model that can run locally.

**About Qwen2.5:**
- Developed by Alibaba Cloud
- Strong performance on various NLP tasks including reasoning
- Supports multiple languages
- Can run efficiently on consumer hardware
- Excellent for agentic applications due to its reasoning capabilities

**Memory Optimization:**
We'll use 4-bit quantization to reduce memory requirements while maintaining good performance.

In [2]:
# Configure model quantization for efficient memory usage
# This reduces model size from ~14GB to ~4GB with minimal performance loss
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Model name - using a smaller variant for better performance
# You can change to "Qwen/Qwen2.5-7B-Instruct" for larger or smaller model
#model_name = "Qwen/Qwen2.5-3B-Instruct"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
#model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"



print(f"Loading model: {model_name}")
print("This may take a few minutes on first run as the model is downloaded...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

print("✓ Model and tokenizer loaded successfully!")
print(f"Model memory footprint: ~{model.get_memory_footprint() / 1e9:.2f} GB")

Loading model: Qwen/Qwen2.5-1.5B-Instruct
This may take a few minutes on first run as the model is downloaded...


Loading weights: 100%|██████████| 338/338 [00:02<00:00, 124.26it/s, Materializing param=model.norm.weight]                              


✓ Model and tokenizer loaded successfully!
Model memory footprint: ~1.12 GB


### Create Text Generation Pipeline

Create a pipeline for easy text generation with the loaded model. This pipeline will be wrapped into a model compatible with smolagents.

In [3]:
# Create a text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.2,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print("✓ Text generation pipeline created successfully!")

# Test the pipeline with a simple query
test_response = llm_pipeline("Hello! What is Python?", max_new_tokens=50)
print("\nTest generation:")
print(test_response[0]['generated_text'][:200] + "...")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'top_p', 'repetition_penalty', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Text generation pipeline created successfully!

Test generation:
Hello! What is Python? I'm a beginner. Can you explain it to me in simple terms?
Python is an easy-to-learn, high-level programming language that's widely used for web development and data analysis.
I...


### Create Smolagents-Compatible Model Wrapper

For smolagents to work with our local model, we need to create a custom model class that wraps our pipeline. This allows the CodeAgent to use our local Qwen model instead of API-based models.

In [4]:
from smolagents.models import MessageRole
from typing import List, Dict, Union, Optional

class Message:
    """Simple message class to hold generated content."""
    def __init__(self, content: str):
        self.content = content
        self.token_usage = None  # smolagents expects this attribute

class LocalLLMModel:
    """
    Wrapper class to make our local LLM compatible with smolagents.
    
    This class implements the interface expected by CodeAgent, allowing us to use
    our locally-hosted Qwen2.5 model instead of API-based models.
    """
    
    def __init__(self, pipeline):
        self.pipeline = pipeline
        
    def __call__(self, messages: List[Union[Dict[str, str], object]], stop_sequences=None, grammar=None) -> str:
        """
        Process messages and return a response.
        
        Args:
            messages: List of message dictionaries or ChatMessage objects with 'role' and 'content'
            stop_sequences: Optional sequences to stop generation
            grammar: Optional grammar constraints (not used with local models)
            
        Returns:
            Generated text response
        """
        # Convert messages to a single prompt string
        # smolagents can pass messages as dicts or ChatMessage objects
        
        prompt_parts = []
        for msg in messages:
            # Handle both dictionary and object formats
            if isinstance(msg, dict):
                role = msg.get("role", "user")
                content = msg.get("content", "")
            else:
                # It's a ChatMessage object with attributes
                role = getattr(msg, "role", "user")
                content = getattr(msg, "content", "")
            
            if role == "system":
                prompt_parts.append(f"System: {content}")
            elif role == "user":
                prompt_parts.append(f"User: {content}")
            elif role == "assistant":
                prompt_parts.append(f"Assistant: {content}")
        
        # Combine all parts into a single prompt
        full_prompt = "\n\n".join(prompt_parts) + "\n\nAssistant:"
        
        # Generate response
        try:
            response = self.pipeline(
                full_prompt,
                max_new_tokens=512,
                temperature=0.1,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.pipeline.tokenizer.eos_token_id
            )
            
            # Extract the generated text (remove the prompt)
            generated_text = response[0]['generated_text']
            # Get only the new generated content
            answer = generated_text[len(full_prompt):].strip()
            
            return answer
        except Exception as e:
            return f"Error generating response: {e}"
    
    def generate(self, messages: List[Union[Dict[str, str], object]], stop_sequences=None, grammar=None) -> Message:
        """
        Generate a response and return it as a Message object.
        
        This method is required by smolagents' MultiStepAgent.
        
        Args:
            messages: List of message dictionaries or ChatMessage objects with 'role' and 'content'
            stop_sequences: Optional sequences to stop generation
            grammar: Optional grammar constraints (not used with local models)
            
        Returns:
            Message object containing the generated text
        """
        # Use the __call__ method to generate the response
        content = self.__call__(messages, stop_sequences, grammar)
        return Message(content)

# Create the model wrapper
agent_model = LocalLLMModel(llm_pipeline)

print("✓ Smolagents-compatible model wrapper created!")
print("✓ The agent can now use our local Qwen2.5 model for reasoning and tool use")


✓ Smolagents-compatible model wrapper created!
✓ The agent can now use our local Qwen2.5 model for reasoning and tool use


## 3 - Load Local Embedding Model (all-MiniLM-L6-v2)

We will load the all-MiniLM-L6-v2 embedding model from Hugging Face. This model converts text into vector representations for similarity search.

**About all-MiniLM-L6-v2:**
- Compact and efficient (only 80MB)
- 384-dimensional embeddings
- Excellent performance for semantic similarity tasks
- Fast inference speed
- Well-suited for RAG applications

In [5]:
# Load the embedding model from Hugging Face
# This model will convert text into 384-dimensional vectors
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {embedding_model_name}")

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cpu'},  # Use 'cuda' if you have a GPU
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)

print("✓ Embedding model loaded successfully!")

# Test the embedding model
test_text = "This is a test sentence."
test_embedding = embedding_model.embed_query(test_text)
print(f"Embedding dimension: {len(test_embedding)}")
print(f"Sample embedding values: {test_embedding[:5]}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 652.74it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded successfully!
Embedding dimension: 384
Sample embedding values: [0.08429647982120514, 0.057953767478466034, 0.00449337437748909, 0.10582111775875092, 0.007083370350301266]


## 4 - Understanding Agentic RAG Architecture

### What is Agentic RAG?

**Agentic RAG** extends traditional RAG by incorporating autonomous decision-making and reasoning capabilities:

1. **Autonomous Tool Use**: The agent decides when and how to use retrieval tools
2. **Multi-Step Reasoning**: Complex queries are broken down into logical steps
3. **Self-Correction**: The agent can evaluate its own outputs and retry if needed
4. **Code Generation**: The agent can write and execute Python code for complex tasks
5. **Transparent Reasoning**: Verbose mode exposes the agent's thought process

### Agentic RAG Workflow with CodeAgent

**Phase 1: Indexing (Same as Traditional RAG)**
1. Load documents from various sources
2. Split documents into smaller chunks
3. Generate embeddings for each chunk
4. Store embeddings in a vector database

**Phase 2: Agentic Querying (Enhanced)**
1. User submits a question
2. **Agent analyzes the question** and formulates a plan
3. **Agent decides to use the retrieval tool** with an appropriate query
4. Agent receives retrieved documents
5. **Agent reasons about the context** - is it sufficient?
6. If needed, agent can retrieve more or refine the query
7. **Agent writes code** to format and process the information
8. Agent generates final answer based on reasoning

### Benefits of CodeAgent

The **CodeAgent** from smolagents has several advantages:

1. **Explicit Reasoning**: Each step is visible in verbose mode
2. **Code-Based Processing**: Can manipulate data programmatically
3. **Tool Orchestration**: Intelligently decides when to use tools
4. **Error Recovery**: Can catch and handle errors gracefully
5. **Extensibility**: Easy to add new tools and capabilities

### Debugging with Verbose Mode

Setting `verbosity_level=2` shows:
- **Task interpretation**: How the agent understands the question
- **Tool selection**: Why the agent chooses specific tools
- **Intermediate results**: Data retrieved and processed
- **Code execution**: Python code written and executed
- **Reasoning steps**: Thought process at each stage
- **Final answer synthesis**: How the answer is constructed

This transparency is invaluable for:
- Understanding agent behavior
- Debugging failures
- Improving prompts and tools
- Building trust in the system

## 5 - Load and Chunk PDF File

We will load the Python Programming PDF file, split it into chunks, and prepare it for embedding.

In [6]:
import os

current_directory = os.getcwd()
print("Current working directory:", current_directory)

Current working directory: c:\Users\I838159\OneDrive - SAP SE\Documents\Files\Personal\AgenticAICert\Samples\02 - Agentic Rag


In [7]:
# Path to the PDF file
pdf_file = "../UnstructureData/PDFFiles/Python Programming.pdf"

print(f"Loading PDF file: {pdf_file}")

# Load the PDF
pdf_loader = PyPDFLoader(pdf_file)

# Create a text splitter for chunking
# Using RecursiveCharacterTextSplitter which intelligently splits on paragraph/sentence boundaries
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,      # Maximum characters per chunk
    chunk_overlap=50,    # Overlap between chunks for context continuity
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Split hierarchy: paragraphs > lines > words > chars
)

# Load and split the PDF
pdf_chunks = pdf_loader.load_and_split(text_splitter)

print(f"✓ PDF loaded and split into {len(pdf_chunks)} chunks")
print(f"\nSample chunk (first 300 characters):")
print(pdf_chunks[0].page_content[:300] + "...")

Loading PDF file: ../UnstructureData/PDFFiles/Python Programming.pdf
✓ PDF loaded and split into 328 chunks

Sample chunk (first 300 characters):
Python ProgrammingHans-Petter Halvorsen
https://www.halvorsen.blog...


## 6 - Load and Chunk Text Files

We will load at least 10 text files from the Python 3.14 documentation directory.

In [8]:
# Path to the text files directory
text_files_dir = "../UnstructureData/TextFiles/python-3.14-docs-text/"

# Get all .txt files in the directory
text_file_paths = glob.glob(os.path.join(text_files_dir, "*.txt"))

print(f"Found {len(text_file_paths)} text files")

# Load at least 10 text files
text_files_to_load = text_file_paths[:min(15, len(text_file_paths))]  # Load 15 files
print(f"Loading {len(text_files_to_load)} text files...")

# Initialize list to store all text chunks
text_chunks = []

# Load each text file
for file_path in text_files_to_load:
    try:
        loader = TextLoader(file_path, encoding='utf-8')
        # Load and split the file
        docs = loader.load()
        chunks = text_splitter.split_documents(docs)
        text_chunks.extend(chunks)
        print(f"  ✓ Loaded: {os.path.basename(file_path)} ({len(chunks)} chunks)")
    except Exception as e:
        print(f"  ✗ Error loading {os.path.basename(file_path)}: {e}")
        # Try with different encoding
        try:
            loader = TextLoader(file_path, encoding='latin-1')
            docs = loader.load()
            chunks = text_splitter.split_documents(docs)
            text_chunks.extend(chunks)
            print(f"  ✓ Loaded with latin-1: {os.path.basename(file_path)} ({len(chunks)} chunks)")
        except Exception as e2:
            print(f"  ✗ Failed with both encodings: {e2}")

print(f"\n✓ Total text chunks: {len(text_chunks)}")
print(f"\nSample text chunk (first 300 characters):")
if text_chunks:
    print(text_chunks[0].page_content[:300] + "...")

Found 8 text files
Loading 8 text files...
  ✓ Loaded: about.txt (3 chunks)
  ✓ Loaded: bugs.txt (9 chunks)
  ✓ Loaded: contents.txt (375 chunks)
  ✓ Loaded: copyright.txt (1 chunks)
  ✓ Loaded: glossary.txt (194 chunks)
  ✓ Loaded: improve-page-nojs.txt (1 chunks)
  ✓ Loaded: improve-page.txt (2 chunks)
  ✓ Loaded: license.txt (155 chunks)

✓ Total text chunks: 740

Sample text chunk (first 300 characters):
About this documentation
************************

Python's documentation is generated from reStructuredText sources
using Sphinx, a documentation generator originally created for Python
and now maintained as an independent project.

Development of the documentation and its toolchain is an entirely
...


## 7 - Load and Chunk HTML Files

We will load at least 10 HTML files from the pandas documentation directory.

In [9]:
# Path to the HTML files directory
html_files_dir = "../UnstructureData/HTMLFiles/pandasDoc"

# Get all .html files in the directory (including subdirectories)
html_file_paths = glob.glob(os.path.join(html_files_dir, "**/*.html"), recursive=True)

print(f"Found {len(html_file_paths)} HTML files")

# Load at least 10 HTML files
html_files_to_load = html_file_paths[:min(15, len(html_file_paths))]  # Load 15 files
print(f"Loading {len(html_files_to_load)} HTML files...")

# Initialize list to store all HTML chunks
html_chunks = []

# Load each HTML file
for file_path in html_files_to_load:
    try:
        loader = UnstructuredHTMLLoader(file_path)
        # Load and split the file
        docs = loader.load()
        chunks = text_splitter.split_documents(docs)
        html_chunks.extend(chunks)
        print(f"  ✓ Loaded: {os.path.basename(file_path)} ({len(chunks)} chunks)")
    except Exception as e:
        print(f"  ✗ Error loading {os.path.basename(file_path)}: {e}")

print(f"\n✓ Total HTML chunks: {len(html_chunks)}")
print(f"\nSample HTML chunk (first 300 characters):")
if html_chunks:
    print(html_chunks[0].page_content[:300] + "...")

Found 3850 HTML files
Loading 15 HTML files...
  ✓ Loaded: 10min.html (1 chunks)
  ✓ Loaded: advanced.html (1 chunks)
  ✓ Loaded: api.html (1 chunks)
  ✓ Loaded: basics.html (1 chunks)
  ✓ Loaded: categorical.html (1 chunks)
  ✓ Loaded: comparison_with_r.html (1 chunks)
  ✓ Loaded: comparison_with_sas.html (1 chunks)
  ✓ Loaded: comparison_with_sql.html (1 chunks)
  ✓ Loaded: comparison_with_stata.html (1 chunks)
  ✓ Loaded: computation.html (1 chunks)
  ✓ Loaded: contributing.html (1 chunks)
  ✓ Loaded: contributing_docstring.html (1 chunks)
  ✓ Loaded: cookbook.html (1 chunks)
  ✓ Loaded: developer.html (1 chunks)
  ✓ Loaded: dsintro.html (1 chunks)

✓ Total HTML chunks: 15

Sample HTML chunk (first 300 characters):
The page has been moved to 10 minutes to pandas...


## 8 - Combine All Document Chunks

Combine all chunks from PDF, Text, and HTML files into a single list for embedding.

In [10]:
# Combine all chunks
all_chunks = pdf_chunks + text_chunks + html_chunks

print(f"Total document chunks from all sources:")
print(f"  - PDF chunks: {len(pdf_chunks)}")
print(f"  - Text chunks: {len(text_chunks)}")
print(f"  - HTML chunks: {len(html_chunks)}")
print(f"  - TOTAL: {len(all_chunks)}")

# Verify we have documents from each type
assert len(pdf_chunks) > 0, "No PDF chunks loaded!"
assert len(text_chunks) >= 10, f"Expected at least 10 text file chunks, got {len(text_chunks)}"
assert len(html_chunks) >= 10, f"Expected at least 10 HTML file chunks, got {len(html_chunks)}"

print("\n✓ All document types successfully loaded and chunked!")

Total document chunks from all sources:
  - PDF chunks: 328
  - Text chunks: 740
  - HTML chunks: 15
  - TOTAL: 1083

✓ All document types successfully loaded and chunked!


## 9 - Create ChromaDB Vector Store

Now we will create embeddings for all document chunks and store them in ChromaDB for efficient retrieval.

**About Vector Databases:**
Vector databases like ChromaDB store document embeddings and enable fast similarity search. When you query the database:
1. Your query is converted to a vector
2. The database finds the most similar document vectors using cosine similarity
3. The top-k most similar documents are returned

This process is much faster than comparing your query to every document linearly.

In [11]:
# Create ChromaDB vector store from documents
# This will:
# 1. Generate embeddings for all chunks using our local embedding model
# 2. Store them in ChromaDB for efficient retrieval

print("Creating vector store with embeddings...")
print(f"This will process {len(all_chunks)} chunks...")
print("This may take a few minutes depending on the number of documents...")

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name='Python_Documentation_Agentic_RAG',
    persist_directory='./chroma_db_agentic'  # Save to disk for reuse
)

print("✓ Vector store created and persisted successfully!")
print(f"Collection name: Python_Documentation_Agentic_RAG")
print(f"Number of documents indexed: {vectorstore._collection.count()}")

Creating vector store with embeddings...
This will process 1083 chunks...
This may take a few minutes depending on the number of documents...
✓ Vector store created and persisted successfully!
Collection name: Python_Documentation_Agentic_RAG
Number of documents indexed: 3249


## 10 - Prepare the Retriever with Agent and Tools

Now we create a custom **Tool** for the agent to use. This tool wraps the ChromaDB retriever and allows the agent to perform semantic search.

### Understanding Tools in Smolagents

**Tools** are functions that agents can call to perform specific tasks. A tool must define:
- **name**: A unique identifier for the tool
- **description**: Explains what the tool does (helps the agent decide when to use it)
- **inputs**: Specifies the parameters the tool accepts
- **output_type**: The type of data the tool returns
- **forward()**: The actual implementation of the tool's functionality

The agent reads tool descriptions and decides which tools to use based on the task at hand. Well-written descriptions are crucial for proper tool selection.

### Retriever Tool Design

Our retriever tool:
1. Takes a query string as input
2. Performs semantic similarity search on the vector database
3. Returns the top-k most relevant document chunks
4. Formats results in a structured way for the agent to process

In [12]:
class RetrieverTool(Tool):
    """
    Custom tool that enables the agent to perform semantic search over the document collection.
    
    This tool wraps the ChromaDB vector store and provides a simple interface for the agent
    to retrieve relevant documents based on a natural language query.
    """
    
    name = "retriever"
    description = """Retrieves relevant Python programming documentation based on a semantic search query.
    Use this tool when you need to find information about Python programming, pandas library, or related topics.
    The tool performs similarity search and returns the most relevant document sections."""
    
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query to match against documentation. Phrase it as a natural language "
                          "statement or question (e.g., 'How do I define a Python function?' or 'pandas DataFrame operations')."
        }
    }
    output_type = "string"

    def __init__(self, vector_db, **kwargs):
        super().__init__(**kwargs)
        self.vector_db = vector_db

    def forward(self, query: str) -> str:
        """
        Execute the retrieval operation.
        
        Args:
            query: Natural language query string
            
        Returns:
            Formatted string containing retrieved documents
        """
        if not isinstance(query, str):
            raise ValueError("Query must be a string.")

        # Perform similarity search - retrieve top 5 most relevant chunks
        results = self.vector_db.similarity_search(query, k=5)
        
        # Format results for the agent
        formatted_docs = "\n".join(
            f"\n--- Document {i+1} ---\n{doc.page_content}" 
            for i, doc in enumerate(results)
        )
        
        return f"Retrieved {len(results)} relevant documents:\n{formatted_docs}"

# Instantiate the retriever tool
retriever_tool = RetrieverTool(vector_db=vectorstore)

print("✓ Retriever tool created successfully!")
print(f"Tool name: {retriever_tool.name}")
print(f"Tool description: {retriever_tool.description[:100]}...")
print("\nThe agent can now use this tool to search the documentation!")

✓ Retriever tool created successfully!
Tool name: retriever
Tool description: Retrieves relevant Python programming documentation based on a semantic search query.
    Use this t...

The agent can now use this tool to search the documentation!


## 11 - Define Agent with Verbose Mode

Now we create the **CodeAgent** with our custom retriever tool. The agent will be able to:
1. Understand user questions
2. Decide when to use the retriever tool
3. Process retrieved documents
4. Generate comprehensive answers
5. Show its reasoning process in verbose mode

### CodeAgent Configuration

**Key Parameters:**
- **tools**: List of tools available to the agent (our retriever tool)
- **model**: The LLM that powers the agent's reasoning (our local Qwen2.5 model)
- **max_steps**: Maximum number of reasoning steps (prevents infinite loops)
- **verbosity_level**: 
  - `0` = Silent (only show final output)
  - `1` = Show major steps
  - `2` = Show detailed reasoning and code execution (recommended for learning)

### Verbose Mode Benefits

With `verbosity_level=2`, you'll see:
- **Task Analysis**: How the agent interprets your question
- **Planning**: The agent's step-by-step plan
- **Tool Calls**: When and why the agent uses the retriever
- **Code Generation**: Python code the agent writes to process data
- **Intermediate Results**: Data at each step
- **Error Handling**: How the agent recovers from issues
- **Final Synthesis**: How the answer is constructed

This transparency is **essential** for:
- Understanding agent behavior
- Debugging unexpected results
- Learning how to design better tools
- Building trust in the system
- Educational purposes

In [13]:
# Create the CodeAgent with verbose mode enabled
agent = CodeAgent(
    tools=[retriever_tool],       # Our custom retriever tool
    model=agent_model,             # Our local Qwen2.5 model
    max_steps=2,                   # Limit reasoning steps for efficiency
    verbosity_level=2,             # Maximum verbosity for full transparency
    planning_interval=2            # Re-plan every 2 steps if needed
)

print("=" * 80)
print("✓ AGENTIC RAG SYSTEM READY!")
print("=" * 80)
print(f"Agent type: CodeAgent")
print(f"Model: Local Qwen2.5-3B-Instruct")
print(f"Tools available: {len(agent.tools)}")
print(f"  - {retriever_tool.name}: {retriever_tool.description[:60]}...")
print(f"Verbosity level: 2 (Full reasoning transparency)")
print(f"Max reasoning steps: 6")
print("\nThe agent will:")
print("  1. Analyze your question and formulate a plan")
print("  2. Decide when to use the retriever tool")
print("  3. Process retrieved documents programmatically")
print("  4. Generate comprehensive answers with reasoning")
print("  5. Show all intermediate steps and code execution")
print("=" * 80)

✓ AGENTIC RAG SYSTEM READY!
Agent type: CodeAgent
Model: Local Qwen2.5-3B-Instruct
Tools available: 2
  - retriever: Retrieves relevant Python programming documentation based on...
Verbosity level: 2 (Full reasoning transparency)
Max reasoning steps: 6

The agent will:
  1. Analyze your question and formulate a plan
  2. Decide when to use the retriever tool
  3. Process retrieved documents programmatically
  4. Generate comprehensive answers with reasoning
  5. Show all intermediate steps and code execution


## 12 - Test the Agent and Analyze Reasoning Process

Now let's test our agentic RAG system with the same queries we used in the simple RAG notebook. 

**What to Observe:**

For each query, pay attention to:
1. **Task Interpretation**: How the agent understands the question
2. **Tool Selection**: Why the agent chooses to use (or not use) the retriever
3. **Query Formulation**: How the agent formulates search queries
4. **Result Processing**: How the agent analyzes retrieved documents
5. **Code Execution**: Any Python code the agent writes
6. **Reasoning Flow**: The logical steps from question to answer
7. **Answer Synthesis**: How the final answer is constructed

**Comparison with Simple RAG:**

Unlike the simple RAG system which follows a fixed pipeline, you'll see the agent:
- Makes explicit decisions about tool usage
- Can refine queries if needed
- Shows transparent reasoning at each step
- Can handle complex multi-step queries more effectively

### Test Query 1: Python Functions

In [14]:
query_1 = "What are Python functions and how do you define them?"

print("=" * 80)
print(f"QUERY 1: {query_1}")
print("=" * 80)
print("\n🤖 AGENT REASONING AND EXECUTION:\n")

# Run the agent with verbose output
# You'll see the agent's complete reasoning process
answer_1 = agent.run(query_1)

print("\n" + "=" * 80)
print("FINAL ANSWER:")
print("=" * 80)
print(answer_1)
print("=" * 80 + "\n")

QUERY 1: What are Python functions and how do you define them?

🤖 AGENT REASONING AND EXECUTION:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are Python functions and how do you define them?                                                           │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

Passing `generation_config` together with generation-related arguments=({'top_p', 'pad_token_id', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### Part 1: Facts Survey

#### **1.1. Facts Given in the Task**

- No additional facts provided directly within the task.

#### **1.2. Facts to Look Up**

To understand what Python functions are:

- Definition of a Python Function
- Syntax for defining a function 
- Examples of different types of functions in Python 

**Where to Find Information:**  
- Online resources such as [Wikipedia](https://en.wikipedia.org/wiki/Function_(computer_science)), official Python 
documentation ([PEP 8 Style Guide](http://peps.python.org), [Python Functions 
Tutorial](https://docs.python-guide.org/scenarios/function.html))  

- Books:
   - *The Definitive Guide to Python* by Mark Lutz.
   - *Learning Python Programming* by David Beazley & Brian Kostlan.

- Documentation websites:
   - Official Python Docs (<https://docs.python.org/>)
   - Stack Overflow (<https://stackoverflow.com/questions/tagged/python-functions>)

- YouTube tutorials/videos with explanations around defining functions.

- Code examples online repositories like GitHub Gists (<https://gist.github.com>)

- Forums and Q&A sites dedicated to learning Python syntax and best practices.

---

### Part 2: Plan

Given the lack of direct factual input, let's proceed using general knowledge and common sense derived through 
research methods outlined previously.

#### Step 0: Gather Knowledge Base

We aim to gather foundational understanding regarding definitions and usage patterns across various aspects of 
Python’s functionality via exploration of existing literature and practical applications.

#### Step 1: Define What We Mean By “Functions”

In terms of computer science theory, a function refers to an operation performed upon data elements during 
execution; they encapsulate computations so that similar actions can be applied repeatedly without repetition. In 
practice, however, many languages offer more nuanced distinctions between procedures, subroutines, routines, 
algorithms, macros, inline assembly code, libraries, modules, packages, classes, objects, instances, interfaces, 
protocols, exceptions, threads, processes, closures, generators, iterators, coroutines, asyncio events, signals, 
callbacks, event loops, asyncio futures, awaitables, context managers, decorators, metaclasses, annotations, type 
hints, abstract base classes, inheritance hierarchies, composition over abstraction principles, polymorphism, duck 
typing, dynamic dispatching, method overriding, operator overloading, generics, functional programming paradigms, 
object-oriented programming concepts, design patterns, dependency injection frameworks, testing methodologies, unit
tests vs integration
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 66.64 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 58.69 seconds]


FINAL ANSWER:
[{'type': 'text', 'text': 'According to our investigation, a Python function is defined as a block of organized, reusable code that performs a particular task. The primary purpose of a function is to simplify your program logic and make it easier to read and maintain.

Key Points:
1. Built-In Functions: These are predefined functions available in Python.
2. Lambda Expressions: They allow you to create small anonymous (non-local) functions at runtime.
3. Anonymous Functions: Similar to lambdas but not necessarily local scope.
4. Decorator Functions: Used to modify behavior before executing another function.
5. Class-Based Methods: Encapsulates related operations inside a class instance.
6. Static/Claass-Level Methods: Can access variables both outside and inside their own scopes.
7. Special Purpose Generator/Coroutine Functionalities: Useful when dealing with large datasets where performance matters.
8. Advanced Features Offered by Modern OOP Constructs: Includes use of @

### Analysis of Agent Behavior - Query 1

**Observe the above output to understand:**

1. **How did the agent interpret the question?**
   - Did it understand this was about Python programming?
   - Did it identify the need for documentation retrieval?

2. **What was the agent's plan?**
   - What steps did it decide to take?
   - Were the steps logical and efficient?

3. **How did the agent use the retriever tool?**
   - What query did it pass to the retriever?
   - Was the query well-formulated?

4. **How did the agent process the results?**
   - Did it write code to analyze the retrieved documents?
   - How did it extract relevant information?

5. **Was the answer grounded in the retrieved context?**
   - Did the agent use information from the documents?
   - Or did it rely on its pre-trained knowledge?

**Key Insight:** The agent's reasoning is transparent, allowing you to debug and improve the system.

### Test Query 2: Python Data Structures

In [15]:
query_2 = "Explain the difference between lists and tuples in Python."

print("=" * 80)
print(f"QUERY 2: {query_2}")
print("=" * 80)
print("\n🤖 AGENT REASONING AND EXECUTION:\n")

answer_2 = agent.run(query_2)

print("\n" + "=" * 80)
print("FINAL ANSWER:")
print("=" * 80)
print(answer_2)
print("=" * 80 + "\n")

QUERY 2: Explain the difference between lists and tuples in Python.

🤖 AGENT REASONING AND EXECUTION:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Explain the difference between lists and tuples in Python.                                                      │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### Part 1: Facts Survey

#### 1.1. Facts Given in the Task
- **Python Lists**: 
  - Definition: A collection of items.
  - Example usage:
    ```python
    my_list = [1, 2, 3]
    ```

  - Operations with lists include indexing (`my_list[0]`), slicing (`my_list[:5]`), concatenation (`my_list + 
another_list`), and membership checks using `in`.

  - List comprehensions allow creating new lists efficiently.

  - Methods such as `.append()`, `.remove()` modify existing lists directly without returning them back to the 
caller.

  - Iterating over elements via loops allows accessing all members sequentially.

  - Sorting methods sort collections but return modified versions rather than sorted copies; they don't change 
original lists unless explicitly called again.

  - Built-in types other than strings support iteration through their characters.

  - Generators produce one element at a time instead of storing everything upfront.

  - Dictionaries store key-value pairs allowing fast access by keys only.

  - Sets eliminate duplicates while preserving uniqueness.

  - Tuples cannot contain mutable objects inside them due to immutability constraints.

  - Immutable vs Mutable Objects:

    *Immutable*: Strings, numbers, booleans, frozen sets, frozensets, constants, dictionaries, classes, modules, 
builtins, type hints, annotations, lambdas, closures, lambda expressions, generator expressions, set literals, 
tuple literals, dict literals, class definitions, module imports, import statements, docstrings, comments, 
whitespace, line breaks, indentation, blank lines, trailing commas, leading underscores, double underscored 
attributes, private variables, protected variables, public properties, getters/setters, constructors/destructors, 
static methods/constructors/deserializers, factory method patterns, singleton pattern implementations, dependency 
injection frameworks, mocking libraries, testing frameworks, logging utilities, database connection pools, session 
management strategies, authentication mechanisms, authorization policies, rate limiting techniques, caching 
strategies, pagination helpers, error handling middleware, response serialization formats, JSON encoding decoders, 
XML parsing serializers, YAML formatting conversions, CSV export/import parsers, SQL queries execution engines, ORM
mappings, data validation rules, security measures, encryption algorithms, hashing methodologies, tokenization 
schemes, password salting practices, two-factor authentication protocols, multi-factor authentication approaches, 
OAuth flows, JWT tokens generation verification expiration date restrictions, CSRF protection tactics, XSS 
prevention countermeasures, web application firewall configurations, content delivery network optimizations
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 4.14 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 41.12 seconds]


FINAL ANSWER:
[{'type': 'text', 'text': 'The primary distinction between lists and tuples in Python revolves around their mutability or lack thereof. Lists can be changed after creation (e.g., adding or removing elements) unlike tuples which remain unchanged once created. This makes tuples more suitable for situations where you want your structure to stay constant throughout its use. For instance, if you need to maintain consistency across multiple operations within a program, utilizing tuples would ensure no unintended changes occur. On the contrary, list modifications could lead to unpredictable behavior depending on future updates made elsewhere in code.']}

This completes our dialogue about the difference between lists and tuples in Python based on the provided information from both parties involved.

If there’s anything else we should cover regarding these concepts, feel free to ask! 😊

---

**Note:** The final text was not included here since it wasn’t part of any previous conve

### Analysis of Agent Behavior - Query 2

**Key Questions to Consider:**

1. **Did the agent's strategy differ from Query 1?**
   - Was the retrieval approach the same or different?
   - Why might the agent have made different decisions?

2. **How did the agent handle comparison questions?**
   - Did it retrieve information about both lists and tuples?
   - How did it structure the comparison?

3. **Code generation (if any):**
   - Did the agent write code to format the comparison?
   - Was the code efficient and correct?

**Learning Point:** Observing how the agent handles different question types helps you understand its capabilities and limitations.

### Test Query 3: Pandas Operations

In [16]:
query_3 = "How do you read a CSV file using pandas and display the first few rows?"

print("=" * 80)
print(f"QUERY 3: {query_3}")
print("=" * 80)
print("\n🤖 AGENT REASONING AND EXECUTION:\n")

answer_3 = agent.run(query_3)

print("\n" + "=" * 80)
print("FINAL ANSWER:")
print("=" * 80)
print(answer_3)
print("=" * 80 + "\n")

QUERY 3: How do you read a CSV file using pandas and display the first few rows?

🤖 AGENT REASONING AND EXECUTION:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How do you read a CSV file using pandas and display the first few rows?                                         │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### Part 1: Facts Survey

#### 1.1. Facts Given in the Task
- **Task Description**: Read a CSV file using pandas and display the first few rows.

No additional facts provided directly within the task description.

#### 1.2. Facts to Look Up
- **Fact Name**: `csv_file_path`
- **Reasoning**:
  To determine the path to the CSV file, one would generally require knowledge of either the local directory 
structure containing the file or an external source providing such details. This fact cannot be derived logically 
without further context regarding how the data was obtained or accessed.

- **Source**: Assume hypothetical API endpoint returning file paths (`/api/files/{file_id}`)

- **Additional Fact Needed**: 
  - `dataframe_columns` — List of column headers expected in the dataframe after reading the csv file.

  Reasoning:
  Knowing what columns exist before loading them allows us to predictably extract only those columns rather than 
all entries, saving time during processing.

- **Source**: Hypothetical RESTful service exposing metadata via `/api/csv_data`

- **Fact Name**: `column_names`
- **Reasoning**:
  Without knowing beforehand exactly what columns are contained in the dataset, attempting to load every row 
indiscriminately results in inefficient memory usage and potentially incorrect behavior due to missing fields being
filled with default values.

- **Source**: Assume hypothetical API endpoint returning column names (`/api/columns`) 

- **Fact Name**: `number_of_rows_to_display`
- **Reasoning**:
  Determining precisely how many rows to show requires understanding both the total number of records in the 
dataset and whether pagination features apply; otherwise, displaying fewer rows than actually loaded wastes 
resources unnecessarily.

- **Source**: Assume hypothetical API endpoint showing full dataset count (`/api/dataset_count`)

- **Fact Name**: `display_format`
- **Reasoning**:
  Understanding desired output format helps decide between printing raw strings vs converting to HTML tables, JSON 
objects, etc.; also determines if formatting options beyond simple text rendering are required.

- **Sources**: Assume endpoints handling different formats:

  - Display Format Endpoint: `/api/display_formatting`
  - Formatting Options Endpoint: `/api/format_options`

- **Fact Name**: `total_number_of_records`
- **Reasoning**:
  Calculating the exact number of records ensures accurate representation regardless of sorting methods applied 
post-loading but pre-sorting does save unnecessary computations since sorted lists consume less space overall 
compared to unsorted versions.
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 63.92 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 21.83 seconds]


FINAL ANSWER:
[{'type': 'text', 'text': 'To read a CSV file using pandas and display the first few rows, you can use the following steps:

1. Specify the location of your CSV file.
2. Use the `pd.read_csv()` function from the pandas library to load the CSV file into a DataFrame.

For example:
```python
import pandas as pd

# Assuming the CSV file is located at '/path/to/myfile.csv'
filename = '/path/to/myfile.csv'

try:
    df = pd.read_csv(filename)
    
    print("First Few Rows:")
    print(df.head())
except Exception as ex:
    print(f"Error occurred while trying to process the file: {ex}")
```

This code snippet demonstrates how to handle potential errors when working with files by catching exceptions and logging any issues encountered during the operation.'
}]



### Analysis of Agent Behavior - Query 3

**Focus Areas:**

1. **Domain shift detection:**
   - Did the agent recognize this question is about pandas (not just Python)?
   - How did this affect its retrieval strategy?

2. **Specificity of retrieval:**
   - What query did the agent use to search for pandas information?
   - Were the retrieved documents relevant to CSV operations?

3. **Code example handling:**
   - If the retrieved documents contained code examples, how did the agent present them?
   - Did the agent generate or format code snippets?

**Insight:** Cross-domain queries (Python + pandas) test the agent's ability to navigate multi-topic documentation.

### Test Query 4: Python Classes

In [17]:
query_4 = "What are Python classes and how do you create them?"

print("=" * 80)
print(f"QUERY 4: {query_4}")
print("=" * 80)
print("\n🤖 AGENT REASONING AND EXECUTION:\n")

answer_4 = agent.run(query_4)

print("\n" + "=" * 80)
print("FINAL ANSWER:")
print("=" * 80)
print(answer_4)
print("=" * 80 + "\n")

QUERY 4: What are Python classes and how do you create them?

🤖 AGENT REASONING AND EXECUTION:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are Python classes and how do you create them?                                                             │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### Part 1: Facts Survey

#### 1.1. Facts Given in the Task
- **Python Classes**: Understanding what they represent.

#### 1.2. Facts to Look Up
- **Definition of Python Class**
- **Syntax for Creating a New Class**

#### 1.3. Facts to Derive
- **Example Code** showing creation of new class.

---

### Part 2: Plan

Given:
- We know what `class` means but don't understand its syntax yet.
- We also lack an example code snippet demonstrating creating a class.

Plan:

1. Search online resources such as Stack Overflow, official Python docs, tutorials, blogs, forums, YouTube videos, 
articles, books, Wikipedia pages, Google Scholar papers, academic journals, technical presentations, webinars, 
conferences, workshops, seminars, industry publications, government websites, company intranets, social media 
groups, LinkedIn discussions, Reddit communities, Quora questions, GitHub repositories, GitLab projects, Bitbucket 
organizations, Docker images, Kubernetes clusters, AWS services, Azure solutions, IBM Cloud platforms, Oracle 
databases, Microsoft SQL Server instances, Apache Hadoop ecosystems, TensorFlow frameworks, PyTorch libraries, 
Scikit-Learn packages, Pandas dataframes, NumPy arrays, Matplotlib plots, Seaborn visualizations, Beautiful Soup 
parsers, BeautifulSoup elements, Requests HTTP client, Selenium WebDriver automation, Flask framework applications,
Django CMS content management systems, AngularJS frontend technologies, React.js JavaScript UI components, Vue.js 
front-end frameworks, Bootstrap CSS utilities, jQuery JS plugins, MongoDB NoSQL database schema design, PostgreSQL 
relational DBMS table structure, MySQL structured storage engines, SQLite lightweight embedded DBMS files, Redis 
key-value store persistence mechanisms, Kafka message brokers streaming architectures, RabbitMQ exchange bindings 
routing strategies, MQTT protocol messaging protocols, RESTful API service endpoints, SOAP vs JSON XML formats 
differences, YAML configuration files structures, CSV comma-separated value format encoding standards, HTML5 markup
languages tags attributes stylesheets, CSS preprocessors variables mixins selectors rules inheritance properties 
typography font-face fonts fallbacks text-shadow blur-radius color-hex codespace rgb hexcodes rgba css-color 
schemes gradients background-image repeat-x repeat-y no-repeat position absolute relative fixed static inline block
flexbox grid layout models dimensions pixels em rem vh vw vcenter center justify align-items items container 
overflow scrollbars resize draggable resizable focusable interactive clickable hover active selected disabled 
readonly readwrite editable focused invalid valid input placeholder auto capitalize camelcase snake case title-case
dasherize slugify underscored hyphenated uppercase
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 2: Duration 2.60 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 9.75 seconds]


FINAL ANSWER:
[{'type': 'text', 'text': "A Python class defines one or more methods (functions) which can be called by name from outside the class itself. The keyword used to define a class is ‘class’. An instance variable stores information about each individual object created using the same class. Methods defined inside a class have access to all other members including private ones."}]



### Comparative Analysis: Agentic RAG vs Simple RAG

**Now that you've seen the agent in action, compare its behavior to the simple RAG system:**

| **Aspect** | **Simple RAG** | **Agentic RAG** |
|------------|----------------|-----------------|
| **Transparency** | Black box generation | Step-by-step reasoning visible |
| **Query Processing** | Fixed embedding of user query | Agent can reformulate queries |
| **Retrieval Strategy** | Always retrieves k documents | Agent decides when to retrieve |
| **Error Recovery** | No self-correction | Can retry with different approach |
| **Complex Queries** | Single-pass processing | Multi-step reasoning possible |
| **Debugging** | Difficult to debug failures | Clear reasoning trail for debugging |
| **Adaptability** | Static pipeline | Dynamic adaptation to query type |

**Key Advantages of Agentic Approach:**

1. **Transparency**: You can see exactly what the agent is doing
2. **Debuggability**: Easy to identify where things go wrong
3. **Flexibility**: Agent adapts to different query types
4. **Learning**: Understanding agent reasoning improves your system design
5. **Trust**: Visible reasoning builds confidence in the system

**When to Use Agentic RAG:**

- Complex, multi-part questions
- Scenarios requiring multi-step reasoning
- When transparency is important
- Educational or research applications
- When you need to debug and improve the system

**When Simple RAG Might Suffice:**

- Simple, straightforward questions
- High-throughput, low-latency requirements
- When a fixed pipeline is well-understood
- Production systems where consistency is critical

## 13 - Define Evaluation Agent for Groundedness

Now we'll create specialized evaluation agents to assess the quality of our answers. The first metric is **Groundedness** (also called **Faithfulness**).

### What is Groundedness?

**Groundedness** measures whether the answer is derived from the provided context (retrieved documents) rather than the model's pre-trained knowledge or hallucinations.

**Why is Groundedness Important?**

In RAG systems, it's crucial that answers are:
- Based on retrieved documents (not made up)
- Factually supported by the context
- Not introducing external information
- Trustworthy and verifiable

**Evaluation Approach:**

We create a specialized agent whose task is to:
1. Analyze the question, retrieved context, and generated answer
2. Identify which parts of the answer come from the context
3. Flag any information not found in the context
4. Assign a groundedness score (1-5)

This agent acts as a "judge" to ensure our main RAG agent is staying grounded in the retrieved documents.

In [18]:
# Create a specialized tool for groundedness evaluation
class GroundednessEvaluator(Tool):
    """
    Tool for evaluating how well an answer is grounded in the provided context.
    
    This tool helps the evaluation agent assess whether the answer is derived
    from the retrieved documents or contains hallucinated information.
    """
    
    name = "groundedness_evaluator"
    description = """Evaluates the groundedness (faithfulness) of an answer relative to the provided context.
    Returns a detailed analysis and score (1-5) indicating how well the answer is derived from the context."""
    
    inputs = {
        "question": {"type": "string", "description": "The original question"},
        "context": {"type": "string", "description": "The retrieved context documents"},
        "answer": {"type": "string", "description": "The answer to evaluate"}
    }
    output_type = "string"

    def __init__(self, llm_pipeline, **kwargs):
        super().__init__(**kwargs)
        self.llm_pipeline = llm_pipeline

    def forward(self, question: str, context: str, answer: str) -> str:
        """Evaluate groundedness of the answer."""
        
        evaluation_prompt = f"""You are an expert evaluator assessing the groundedness of AI-generated answers.

**Task**: Evaluate how well the answer is grounded in (derived from) the provided context.

**Question**: {question}

**Context**:
{context}

**Answer to Evaluate**:
{answer}

**Evaluation Criteria** (1-5 scale):
1 - Not grounded: Answer is not based on the context at all
2 - Minimally grounded: Only small portions come from the context
3 - Moderately grounded: Answer is partially based on context
4 - Mostly grounded: Most of the answer comes from context
5 - Completely grounded: Answer is entirely derived from context

**Instructions**:
1. Identify key claims in the answer
2. Check if each claim is supported by the context
3. Flag any information not found in the context
4. Consider if the answer introduces external knowledge
5. Assign a score and explain your reasoning

**Provide**:
- Detailed analysis (2-3 sentences)
- List of grounded vs. ungrounded claims
- Final score (1-5)
"""
        
        try:
            response = self.llm_pipeline(
                evaluation_prompt,
                max_new_tokens=400,
                temperature=0.1,
                do_sample=True
            )
            result = response[0]['generated_text'][len(evaluation_prompt):].strip()
            return f"Groundedness Evaluation:\n{result}"
        except Exception as e:
            return f"Error in evaluation: {e}"

# Create the groundedness evaluator tool
groundedness_tool = GroundednessEvaluator(llm_pipeline=llm_pipeline)

# Create an evaluation agent specifically for groundedness assessment
groundedness_agent = CodeAgent(
    tools=[groundedness_tool],
    model=agent_model,
    max_steps=4,
    verbosity_level=2
)

print("✓ Groundedness evaluation agent created!")
print("This agent will assess whether answers are grounded in retrieved context.")

✓ Groundedness evaluation agent created!
This agent will assess whether answers are grounded in retrieved context.


## 14 - Test Groundedness Evaluation Agent

Let's evaluate the groundedness of our previous answers. We'll assess whether the agent's answers were truly based on the retrieved documents.

### What to Look For:

1. **Evaluation Process**: How does the evaluation agent analyze the answer?
2. **Claim Identification**: Does it break down the answer into verifiable claims?
3. **Context Matching**: How does it match claims to the context?
4. **Score Justification**: Is the score well-reasoned?
5. **Constructive Feedback**: Does it identify specific issues?

**Note:** For this evaluation, we need to retrieve the actual context that was used by the main agent. We'll manually retrieve it using the same query.

In [19]:
# Evaluate Query 2: Lists vs Tuples
print("=" * 80)
print("EVALUATING GROUNDEDNESS FOR QUERY 2")
print("=" * 80)

# Retrieve the context that would have been used
context_docs_2 = vectorstore.similarity_search(query_2, k=5)
context_2 = "\n\n".join([f"--- Document {i+1} ---\n{doc.page_content}" for i, doc in enumerate(context_docs_2)])

# Prepare evaluation request
eval_request_2 = f"""Evaluate the groundedness of this answer:

Question: {query_2}

Context:
{context_2[:2000]}... [truncated for brevity]

Answer:
{answer_2}

Please assess how well this answer is grounded in the provided context.
"""

print("\n🤖 GROUNDEDNESS EVALUATION AGENT:\n")
groundedness_eval_2 = groundedness_agent.run(eval_request_2)

print("\n" + "=" * 80)
print("GROUNDEDNESS EVALUATION RESULT:")
print("=" * 80)
print(groundedness_eval_2)
print("=" * 80 + "\n")

EVALUATING GROUNDEDNESS FOR QUERY 2

🤖 GROUNDEDNESS EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the groundedness of this answer:                                                                       │
│                                                                                                                 │
│ Question: Explain the difference between lists and tuples in Python.                                            │
│                                                                                                                 │
│ Context:                                                                                                        │
│ --- Document 1 ---                                                                                              │
│ * Why does Python use methods for some functionality (e.g.                                                      │
│       list.index()) but functions for other (e.g. len(list))?                                                   │
│                                                                                                                 │
│     * Why is join() a string method instead of a list or tuple method?                                          │
│                                                                                                                 │
│     * How fast are exceptions?                                                                                  │
│                                                                                                                 │
│     * Why isn't there a switch or case statement in Python?                                                     │
│                                                                                                                 │
│     * Can't you emulate threads in the interpreter instead of relying                                           │
│       on an OS-specific thread implementation?                                                                  │
│                                                                                                                 │
│     * Why can't lambda expressions contain statements?                                                          │
│                                                                                                                 │
│ --- Document 2 ---                                                                                              │
│ * Why does Python use methods for some functionality (e.g.                                                      │
│       list.index()) but functions for other (e.g. len(list))?                                                   │
│                                                                                                                 │
│     * Why is join() a string method instead of a list or tuple method?                                          │
│                                                                                                                 │
│     * How fast are exceptions?                                                                                  │
│                                                                                                                 │
│     * Why isn't there a switch or case statement in Python?                                                     │
│                                                                                                                 │
│     * Can't you emulate threads in the interpreter instead of relying                                           │
│       on an OS-specific thread implementation?                                                                  │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To determine whether the answer is grounded in the provided context, let's analyze the relevant parts:             
                                                                                                                   
### Context Analysis:                                                                                              
From Documents 1 through 4, several questions discuss why certain features or functionalities are implemented      
differently in Python compared to similar constructs used in other languages.                                      
                                                                                                                   
For example:                                                                                                       
- Question: *"Why doesn't Python support switches/cases?"                                                          
- Answer: *"Because they're slower."                                                                               
                                                                                                                   
- Question: *"Can't you implement threading yourself?"                                                             
- Answer: *"Yes, but it relies on system-level threading."                                                         
                                                                                                                   
However, none of these discussions specifically address the fundamental differences between lists and tuples       
themselves. Instead, they focus on general language features such as control structures, concurrency models, etc.  
                                                                                                                   
### Relevant Parts Mentioned:                                                                                      
None of the contexts explicitly mention the specific topic related to lists vs. tuples discussed in the            
user-provided answer.                                                                                              
                                                                                                                   
Therefore, while the overall discussion covers various aspects of Python syntax and semantics, the direct          
comparison between lists and tuples lacks explicit reference to those topics.                                      
                                                                                                                   
### Conclusion:                                                                                                    
Given the absence of references to lists versus tuples specifically addressed in the provided documentation, the   
answer cannot claim grounding solely due to the content[3

...

[Step 4: Duration 25.02 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 50.77 seconds]


GROUNDEDNESS EVALUATION RESULT:
[{'type': 'text', 'text': "Your response seems correct given the current understanding of the document. Let me reiterate what needs to be understood by users when answering questions involving lists and tuples:

1. **Mutability**: Understand that `list` allows changing data types after instantiation, whereas `tuple` retains its original state.
   
   ```python
   # Correct usage:
   lst = []
   lst.append(1)
   print(lst)  # Output will vary each run

   tup = ()
   tup += ('a',)
   print(tup)  # Always prints ('a')
   ```

2. **Use Cases**: Recognize scenarios where using `list` might cause issues because of potential mutation.

   ```python
   class MyClass(object):
       @classmethod
       def mutate(cls):
           cls.data_list = []  # Mutating existing list object leads to unexpected results
           
   >>> obj.mutate()
   TypeError: unsupported operand type(s) for +: 'int' and 'MyClass'
   ```
  
3. **Immutable Types**: Familiarise oneself 

In [20]:
# Evaluate Query 3: Pandas CSV Operations
print("=" * 80)
print("EVALUATING GROUNDEDNESS FOR QUERY 3")
print("=" * 80)

# Retrieve the context
context_docs_3 = vectorstore.similarity_search(query_3, k=5)
context_3 = "\n\n".join([f"--- Document {i+1} ---\n{doc.page_content}" for i, doc in enumerate(context_docs_3)])

# Prepare evaluation request
eval_request_3 = f"""Evaluate the groundedness of this answer:

Question: {query_3}

Context:
{context_3[:2000]}... [truncated for brevity]

Answer:
{answer_3}

Please assess how well this answer is grounded in the provided context.
"""

print("\n🤖 GROUNDEDNESS EVALUATION AGENT:\n")
groundedness_eval_3 = groundedness_agent.run(eval_request_3)

print("\n" + "=" * 80)
print("GROUNDEDNESS EVALUATION RESULT:")
print("=" * 80)
print(groundedness_eval_3)
print("=" * 80 + "\n")

EVALUATING GROUNDEDNESS FOR QUERY 3

🤖 GROUNDEDNESS EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the groundedness of this answer:                                                                       │
│                                                                                                                 │
│ Question: How do you read a CSV file using pandas and display the first few rows?                               │
│                                                                                                                 │
│ Context:                                                                                                        │
│ --- Document 1 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 2 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 3 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 4 ---                                                                                              │
│ The page has been moved to pandas docstring guide                                                               │
│                                                                                                                 │
│ --- Document 5 ---                                                                                              │
│ The page has been moved to pandas docstring guide... [truncated for brevity\]                                   │
│                                                                                                                 │
│ Answer:                                                                                                         │
│ [{'type': 'text', 'text': 'To read a CSV file using pandas and display the first few rows, you can use the      │
│ following steps:                                                                                                │
│                                                                                                                 │
│ 1. Specify the location of your CSV file.                                                                       │
│ 2. Use the `pd.read_csv()` function from the pandas library to load the CSV file into a DataFrame.              │
│                                                                                                                 │
│ For example:                                                                                                    │
│ ```python                                                                                                       │
│ import pandas as pd                                                                                             │
│                                                                                                                 │
│ # Assuming the CSV file is located at '/path/to/myfile.csv'                                                     │
│ filename = '/path/to/myfile.csv'                      

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To determine whether the answer is grounded within the provided context, we need to analyze the content of the     
different documents.                                                                                               
                                                                                                                   
### Analysis:                                                                                                      
1. **Document 1**: This document contains text related to moving a webpage.                                        
2. **Document 2** - Same as above                                                                                  
3. **Document 3** - Also mentions moving a webpage.                                                                
4. **Document 4** - Contains instructions regarding reading a CSV file using pandas.                               
5. **Document 5** - Similar to Document 4                                                                          
                                                                                                                   
From Documents 1 through 5, there's no mention of handling CSV files or displaying row headers. Therefore, none of 
these documents contain relevant information pertaining to reading a CSV file using pandas.                        
                                                                                                                   
Given this lack of relevance, let’s conclude:                                                                      
                                                                                                                   
### Conclusion:                                                                                                    
Based on the absence of pertinent documentation discussing CSV processing via pandas, the provided answer appears  
ungrounded within the context presented here.                                                                      
                                                                                                                   
Therefore, the evaluation concludes that the answer lacks grounding due to insufficient contextual references.     
Thus, the assessment would rate low based on the criteria outlined.                                                

...

[Step 4: Duration 41.61 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 10.38 seconds]


GROUNDEDNESS EVALUATION RESULT:
{'answer': 'The given response does not directly relate to evaluating the groundedness of another answer. Instead, it provides detailed guidance on how to read a CSV file using pandas.'}

Assistant:
{
"type": "text",
"text": "The provided answer discusses methods for reading a CSV file using pandas without referencing the original question or its context."
}



### Insights from Groundedness Evaluation

**Key Observations:**

1. **Multi-Agent Architecture**: We now have specialized agents for different tasks:
   - Main RAG agent for answering questions
   - Evaluation agent for assessing groundedness

2. **Agent as Judge**: The evaluation agent acts as an objective judge, analyzing the main agent's output

3. **Transparency**: The evaluation shows exactly which parts of the answer are grounded

4. **Quality Control**: This helps identify when the main agent is hallucinating or introducing external knowledge

**Practical Applications:**

- **Development**: Use evaluation agents to improve your main system
- **Quality Assurance**: Automatically flag low-groundedness answers
- **User Trust**: Provide users with confidence scores
- **Continuous Improvement**: Collect groundedness scores to track system performance over time

## 15 - Define Evaluation Agent for Context Relevance

The second evaluation metric is **Context Relevance**.

### What is Context Relevance?

**Context Relevance** measures whether the retrieved documents actually contain information needed to answer the question.

**Why is Context Relevance Important?**

Even if an answer is perfectly grounded, it's only as good as the context it's based on. Poor retrieval leads to:
- Incomplete answers
- Irrelevant information
- Missed key details
- Poor user experience

**What We're Evaluating:**

This metric specifically assesses the retrieval quality:
- Are the retrieved documents relevant to the question?
- Do they contain the information needed to answer fully?
- Is there important information missing?
- How much irrelevant content was retrieved?

**Evaluation Approach:**

The context relevance agent:
1. Analyzes the question to determine what information is needed
2. Reviews the retrieved documents
3. Assesses whether they contain the necessary information
4. Identifies gaps or irrelevant content
5. Assigns a relevance score (1-5)

This helps us understand if our retrieval strategy is effective.

In [21]:
# Create a specialized tool for context relevance evaluation
class ContextRelevanceEvaluator(Tool):
    """
    Tool for evaluating the relevance of retrieved context to the question.
    
    This tool helps assess whether the retrieval system is finding the right documents
    to answer the user's question effectively.
    """
    
    name = "context_relevance_evaluator"
    description = """Evaluates how relevant the retrieved context is to answering the question.
    Returns a detailed analysis and score (1-5) indicating context quality."""
    
    inputs = {
        "question": {"type": "string", "description": "The original question"},
        "context": {"type": "string", "description": "The retrieved context documents to evaluate"}
    }
    output_type = "string"

    def __init__(self, llm_pipeline, **kwargs):
        super().__init__(**kwargs)
        self.llm_pipeline = llm_pipeline

    def forward(self, question: str, context: str) -> str:
        """Evaluate relevance of the retrieved context."""
        
        evaluation_prompt = f"""You are an expert evaluator assessing the relevance of retrieved documents.

**Task**: Evaluate how relevant the retrieved context is for answering the question.

**Question**: {question}

**Retrieved Context**:
{context}

**Evaluation Criteria** (1-5 scale):
1 - Not relevant: Context doesn't help answer the question at all
2 - Minimally relevant: Context contains little useful information
3 - Moderately relevant: Context is somewhat helpful but incomplete
4 - Mostly relevant: Context contains most of the needed information
5 - Highly relevant: Context is perfectly suited to answer the question

**Instructions**:
1. Identify what information is needed to answer the question
2. Check if the context contains that information
3. Assess the proportion of relevant vs. irrelevant content
4. Identify any critical missing information
5. Assign a score and explain your reasoning

**Provide**:
- Analysis of information needs (1-2 sentences)
- Assessment of context coverage (1-2 sentences)
- Identified gaps or irrelevant content
- Final score (1-5)
"""
        
        try:
            response = self.llm_pipeline(
                evaluation_prompt,
                max_new_tokens=400,
                temperature=0.1,
                do_sample=True
            )
            result = response[0]['generated_text'][len(evaluation_prompt):].strip()
            return f"Context Relevance Evaluation:\n{result}"
        except Exception as e:
            return f"Error in evaluation: {e}"

# Create the context relevance evaluator tool
context_relevance_tool = ContextRelevanceEvaluator(llm_pipeline=llm_pipeline)

# Create an evaluation agent specifically for context relevance assessment
context_relevance_agent = CodeAgent(
    tools=[context_relevance_tool],
    model=agent_model,
    max_steps=4,
    verbosity_level=2
)

print("✓ Context relevance evaluation agent created!")
print("This agent will assess whether retrieved documents are relevant to the question.")

✓ Context relevance evaluation agent created!
This agent will assess whether retrieved documents are relevant to the question.


## 16 - Test Context Relevance Evaluation Agent

Let's evaluate how well our retriever is performing. Are we getting the right documents for each question?

### What This Tells Us:

1. **Retrieval Quality**: How good is our embedding-based similarity search?
2. **Chunk Size Appropriateness**: Are our 512-character chunks the right size?
3. **Coverage**: Are we indexing the right documents?
4. **Query Formulation**: Would different query phrasing improve results?

**Insight:** Low context relevance scores indicate retrieval problems, not generation problems.

In [22]:
# Evaluate Context Relevance for Query 2
print("=" * 80)
print("EVALUATING CONTEXT RELEVANCE FOR QUERY 2")
print("=" * 80)

# Use the same context we retrieved earlier
eval_context_request_2 = f"""Evaluate the relevance of the retrieved context:

Question: {query_2}

Retrieved Context:
{context_2[:2000]}... [truncated for brevity]

Please assess how relevant this context is for answering the question.
"""

print("\n🤖 CONTEXT RELEVANCE EVALUATION AGENT:\n")
context_relevance_eval_2 = context_relevance_agent.run(eval_context_request_2)

print("\n" + "=" * 80)
print("CONTEXT RELEVANCE EVALUATION RESULT:")
print("=" * 80)
print(context_relevance_eval_2)
print("=" * 80 + "\n")

EVALUATING CONTEXT RELEVANCE FOR QUERY 2

🤖 CONTEXT RELEVANCE EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the relevance of the retrieved context:                                                                │
│                                                                                                                 │
│ Question: Explain the difference between lists and tuples in Python.                                            │
│                                                                                                                 │
│ Retrieved Context:                                                                                              │
│ --- Document 1 ---                                                                                              │
│ * Why does Python use methods for some functionality (e.g.                                                      │
│       list.index()) but functions for other (e.g. len(list))?                                                   │
│                                                                                                                 │
│     * Why is join() a string method instead of a list or tuple method?                                          │
│                                                                                                                 │
│     * How fast are exceptions?                                                                                  │
│                                                                                                                 │
│     * Why isn't there a switch or case statement in Python?                                                     │
│                                                                                                                 │
│     * Can't you emulate threads in the interpreter instead of relying                                           │
│       on an OS-specific thread implementation?                                                                  │
│                                                                                                                 │
│     * Why can't lambda expressions contain statements?                                                          │
│                                                                                                                 │
│ --- Document 2 ---                                                                                              │
│ * Why does Python use methods for some functionality (e.g.                                                      │
│       list.index()) but functions for other (e.g. len(list))?                                                   │
│                                                                                                                 │
│     * Why is join() a string method instead of a list or tuple method?                                          │
│                                                                                                                 │
│     * How fast are exceptions?                                                                                  │
│                                                                                                                 │
│     * Why isn't there a switch or case statement in Python?                                                     │
│                                                                                                                 │
│     * Can't you emulate threads in the interpreter instead of relying                                           │
│       on an OS-specific thread implementation?                                                                  │
│                                                       

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To determine the relevance of the retrieved context to the question, we need to analyze whether the content        
addresses the differences between lists and tuples.                                                                
                                                                                                                   
### Analysis:                                                                                                      
1. **Document 1**: Discusses why Python uses different approaches for certain functionalities.                     
   - This doesn’t address the specific topic of lists vs. tuples.                                                  
                                                                                                                   
2. **Document 2**: Similar discussion as Document 1.                                                               
   - Again, irrelevant to discussing lists versus tuples.                                                          
                                                                                                                   
3. **Document 3** and **Document 4**: Both discuss arrays and loops in Python.                                     
   - These also aren’t related to explaining the distinction between lists and tuples.                             
                                                                                                                   
Given the above points, none of the documents specifically focus on comparing lists and tuples. Therefore, the     
context is generally unrelated to addressing the question regarding the difference between lists and tuples.       
                                                                                                                   
### Conclusion:                                                                                                    
Based on the analysis, the context is **not very relevant** to the question asked.                                 
                                                                                                                   
Final Answer: Not Relevant                                                                                         
                                                                                                                   
```python                                                                                                          
# No action required here since no further processing is necessary based on the logic derived.                     
```                                                                                                                
                                                                                                                   
This concludes the evaluation process; however, please note that while the solution adhered strictly to the        
guidelines, the actual outcome would depend entirely upon the accuracy of the initial [38;2;23

...

[Step 4: Duration 26.49 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 25.60 seconds]


CONTEXT RELEVANCE EVALUATION RESULT:
[{'type': 'text', 'text': "The provided context discusses various topics including Python features, exception handling, threading, and arrays. It primarily focuses on general aspects of Python rather than directly addressing the concept of lists and tuples."}]
Assistant: {'type': 'text', 'text': "The context provides information about Python concepts beyond just lists and tuples, making it less relevant to the given question asking about their distinctiveness."}
Assistant: {'type': 'text', 'text': "Yes, the context is somewhat relevant because it touches on several important areas of Python development, but its primary subject matter diverges from the direct comparison needed by the question concerning lists and tuples."}

Assistant: {'type': 'text', 'text': "No, the context is not sufficiently focused on lists and tuples to be considered highly relevant to the question seeking clarification on their distinctions."}



In [23]:
# Evaluate Context Relevance for Query 3
print("=" * 80)
print("EVALUATING CONTEXT RELEVANCE FOR QUERY 3")
print("=" * 80)

eval_context_request_3 = f"""Evaluate the relevance of the retrieved context:

Question: {query_3}

Retrieved Context:
{context_3[:2000]}... [truncated for brevity]

Please assess how relevant this context is for answering the question.
"""

print("\n🤖 CONTEXT RELEVANCE EVALUATION AGENT:\n")
context_relevance_eval_3 = context_relevance_agent.run(eval_context_request_3)

print("\n" + "=" * 80)
print("CONTEXT RELEVANCE EVALUATION RESULT:")
print("=" * 80)
print(context_relevance_eval_3)
print("=" * 80 + "\n")

EVALUATING CONTEXT RELEVANCE FOR QUERY 3

🤖 CONTEXT RELEVANCE EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the relevance of the retrieved context:                                                                │
│                                                                                                                 │
│ Question: How do you read a CSV file using pandas and display the first few rows?                               │
│                                                                                                                 │
│ Retrieved Context:                                                                                              │
│ --- Document 1 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 2 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 3 ---                                                                                              │
│ The page has been moved to 10 minutes to pandas                                                                 │
│                                                                                                                 │
│ --- Document 4 ---                                                                                              │
│ The page has been moved to pandas docstring guide                                                               │
│                                                                                                                 │
│ --- Document 5 ---                                                                                              │
│ The page has been moved to pandas docstring guide... [truncated for brevity\]                                   │
│                                                                                                                 │
│ Please assess how relevant this context is for answering the question.                                          │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To determine the relevancy of the retrieved context for the question, we need to analyze whether the content       
pertains to reading a CSV file using pandas.                                                                       
                                                                                                                   
### Step-by-step Analysis:                                                                                         
                                                                                                                   
**Step 1:** Identify key terms related to the question.                                                            
   - Key term(s): Read CSV file                                                                                    
                                                                                                                   
**Step 2:** Analyze each document for presence of the identified key term.                                         
                                                                                                                   
#### Document 1:                                                                                                   
```                                                                                                                
Document 1                                                                                                         
```                                                                                                                
                                                                                                                   
#### Document 2:                                                                                                   
```                                                                                                                
Document 2                                                                                                         
```                                                                                                                
                                                                                                                   
#### Document 3:                                                                                                   
```                                                                                                                
Document 3                                                                                                         
```                                                                                                                
                                                                                                                   
#### Document 4:                                                                                                   
```                                                                                                                
pandas docstring guide...                                                                                          
```                                                                                                                
                                                                                                                   
#### Document 5:                                                                                                   
```                                                                                                                
pandas docstring guide...                                                                                          
```                                                     

...

[Step 4: Duration 2.71 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 10.79 seconds]


CONTEXT RELEVANCE EVALUATION RESULT:
{'final_answer': 'The provided context does not directly address or relate to the topic of reading a CSV file using pandas as described by the question. The majority of the texts discuss moving pages rather than discussing data manipulation techniques such as reading from files like CSV.', 'level_of_relevance': 'Very Low'},]



### Insights from Context Relevance Evaluation

**What This Metric Reveals:**

1. **Retrieval Effectiveness**: Are we finding the right documents?
2. **Embedding Quality**: Does semantic similarity work well for our domain?
3. **Document Coverage**: Did we index the right materials?

**Improving Context Relevance:**

If scores are low, consider:
- **Adjust chunk size**: Try 1024 or 256 characters
- **Increase k**: Retrieve more documents (e.g., k=10)
- **Hybrid search**: Combine semantic and keyword search
- **Query expansion**: Rephrase or expand queries
- **Better embeddings**: Use domain-specific embedding models
- **Add more documents**: Expand the knowledge base

**Key Distinction:**
- **Low Context Relevance + High Groundedness** = Good generation, poor retrieval
- **High Context Relevance + Low Groundedness** = Good retrieval, hallucinating generator

## 17 - Define Evaluation Agent for Answer Completeness

The third evaluation metric is **Answer Completeness**.

### What is Answer Completeness?

**Answer Completeness** measures whether the answer fully addresses all aspects of the question.

**Why is Answer Completeness Important?**

Even with good context and grounded answers, the system can still fail by:
- Answering only part of the question
- Missing important nuances
- Being overly brief
- Ignoring implications or follow-up aspects

**What We're Evaluating:**

This metric assesses the answer quality:
- Does it address all parts of the question?
- Is it sufficiently detailed?
- Does it provide examples when appropriate?
- Are there important aspects left unaddressed?

**Evaluation Approach:**

The completeness agent:
1. Breaks down the question into components
2. Checks if each component is addressed in the answer
3. Assesses the depth and detail of the response
4. Identifies missing information or aspects
5. Assigns a completeness score (1-5)

This helps ensure our answers are comprehensive and useful.

In [24]:
# Create a specialized tool for answer completeness evaluation
class CompletenessEvaluator(Tool):
    """
    Tool for evaluating the completeness of an answer relative to the question.
    
    This tool helps assess whether the answer fully addresses all aspects of the
    user's question in sufficient detail.
    """
    
    name = "completeness_evaluator"
    description = """Evaluates how completely an answer addresses all aspects of the question.
    Returns a detailed analysis and score (1-5) indicating answer completeness."""
    
    inputs = {
        "question": {"type": "string", "description": "The original question"},
        "answer": {"type": "string", "description": "The answer to evaluate"}
    }
    output_type = "string"

    def __init__(self, llm_pipeline, **kwargs):
        super().__init__(**kwargs)
        self.llm_pipeline = llm_pipeline

    def forward(self, question: str, answer: str) -> str:
        """Evaluate completeness of the answer."""
        
        evaluation_prompt = f"""You are an expert evaluator assessing the completeness of AI-generated answers.

**Task**: Evaluate how completely the answer addresses all aspects of the question.

**Question**: {question}

**Answer to Evaluate**:
{answer}

**Evaluation Criteria** (1-5 scale):
1 - Incomplete: Addresses very little of the question
2 - Partially complete: Addresses some aspects but misses major points
3 - Moderately complete: Addresses main points but lacks detail or minor aspects
4 - Mostly complete: Addresses nearly all aspects with good detail
5 - Fully complete: Comprehensively addresses all aspects with appropriate detail

**Instructions**:
1. Break down the question into key aspects/components
2. Check if each aspect is addressed in the answer
3. Assess the depth and detail of coverage for each aspect
4. Identify any missing information or under-addressed points
5. Assign a score and explain your reasoning

**Provide**:
- List of question components (bullet points)
- Coverage assessment for each component
- Missing or under-addressed aspects
- Final score (1-5)
"""
        
        try:
            response = self.llm_pipeline(
                evaluation_prompt,
                max_new_tokens=400,
                temperature=0.1,
                do_sample=True
            )
            result = response[0]['generated_text'][len(evaluation_prompt):].strip()
            return f"Answer Completeness Evaluation:\n{result}"
        except Exception as e:
            return f"Error in evaluation: {e}"

# Create the completeness evaluator tool
completeness_tool = CompletenessEvaluator(llm_pipeline=llm_pipeline)

# Create an evaluation agent specifically for answer completeness assessment
completeness_agent = CodeAgent(
    tools=[completeness_tool],
    model=agent_model,
    max_steps=4,
    verbosity_level=2
)

print("✓ Answer completeness evaluation agent created!")
print("This agent will assess whether answers fully address all aspects of questions.")

✓ Answer completeness evaluation agent created!
This agent will assess whether answers fully address all aspects of questions.


## 18 - Test Answer Completeness Evaluation Agent

Let's evaluate how complete our answers are. Do they fully address what the user asked?

### What This Tells Us:

1. **Answer Quality**: Is the system providing useful, complete responses?
2. **Generation Strategy**: Is the agent being thorough enough?
3. **User Satisfaction**: Would users need to ask follow-up questions?

**Completeness vs. Verbosity:**
- A complete answer addresses all aspects without unnecessary detail
- A verbose answer may be long but still miss key points
- The goal is comprehensive yet concise coverage

In [25]:
# Evaluate Answer Completeness for Query 1
print("=" * 80)
print("EVALUATING ANSWER COMPLETENESS FOR QUERY 1")
print("=" * 80)

eval_completeness_request_1 = f"""Evaluate the completeness of this answer:

Question: {query_1}

Answer:
{answer_1}

Please assess how completely this answer addresses all aspects of the question.
"""

print("\n🤖 ANSWER COMPLETENESS EVALUATION AGENT:\n")
completeness_eval_1 = completeness_agent.run(eval_completeness_request_1)

print("\n" + "=" * 80)
print("ANSWER COMPLETENESS EVALUATION RESULT:")
print("=" * 80)
print(completeness_eval_1)
print("=" * 80 + "\n")

EVALUATING ANSWER COMPLETENESS FOR QUERY 1

🤖 ANSWER COMPLETENESS EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the completeness of this answer:                                                                       │
│                                                                                                                 │
│ Question: What are Python functions and how do you define them?                                                 │
│                                                                                                                 │
│ Answer:                                                                                                         │
│ [{'type': 'text', 'text': 'According to our investigation, a Python function is defined as a block of           │
│ organized, reusable code that performs a particular task. The primary purpose of a function is to simplify your │
│ program logic and make it easier to read and maintain.                                                          │
│                                                                                                                 │
│ Key Points:                                                                                                     │
│ 1. Built-In Functions: These are predefined functions available in Python.                                      │
│ 2. Lambda Expressions: They allow you to create small anonymous (non-local) functions at runtime.               │
│ 3. Anonymous Functions: Similar to lambdas but not necessarily local scope.                                     │
│ 4. Decorator Functions: Used to modify behavior before executing another function.                              │
│ 5. Class-Based Methods: Encapsulates related operations inside a class instance.                                │
│ 6. Static/Claass-Level Methods: Can access variables both outside and inside their own scopes.                  │
│ 7. Special Purpose Generator/Coroutine Functionalities: Useful when dealing with large datasets where           │
│ performance matters.                                                                                            │
│ 8. Advanced Features Offered by Modern OOP Constructs: Includes use of @staticmethod/@classmethod decorators,   │
│ property attributes, descriptor protocol implementation, slots management, super() call mechanism, and          │
│ async/await keywords among others.                                                                              │
│                                                                                                                 │
│ For reference, below is a simple python snippet showing one way to declare a function:                          │
│                                                                                                                 │
│ ```python                                                                                                       │
│ def hello_world():                                                                                              │
│     """                                                                                                         │
│     Prints Hello World!                                                                                         │
│                                                                                                                 │
│     >>> hello_world()                                                                                           │
│     Hello World!                                                                                                │
│                                                                                                                 │
│     :param name: Name passed along with output string 

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
{'type': 'text', 'text': 'The answer provides comprehensive details regarding different types of Python functions  
such as built-in functions, lambda expressions, anonymous functions, decorator functions, class-based methods,     
static/class-level methods, special functional features offered through modern constructs, and advanced            
functionalities supported by Python's syntax. It also includes practical example codes demonstrating various ways  
to implement functions.                                                                                            
                                                                                                                   
However, there could still be some improvements:                                                                   
1. Lack of emphasis on advantages over traditional languages.                                                      
2. No mention of global/local scoping differences.                                                                 
3. Minimal discussion on dynamic typing vs statically typed programming concepts.'                                 
                                                                                                                   
Final Answer: This answer comprehensively covers most key points concerning Python functions while addressing      
potential gaps in understanding.'}                                                                                 
                                                                                                                   
Thought: First, let us understand the context - we're evaluating the completeness of the previous answer against   
the actual question asked ("What are Python functions and how do you define them?"). We'll consider factors like   
clarity, depth, relevance, accuracy, and lack thereof.                                                             
                                                                                                                   
Code:                                                                                                              
                                                                                                                   
```python                                                                                                          
def evaluation(complete_answer):                                                                                   
    """                                                                                                            
    Evaluates whether the complete answer fully answers the question posed.                                        
                                                                                                                   
    Parameters:                                                                                                    
 [38;2;230;237;243;48;2;13;17;23

...

[Step 4: Duration 3.79 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 9.36 seconds]


ANSWER COMPLETENESS EVALUATION RESULT:
{'answer': 'This answer does address the main point about what Python functions are and why they're useful. However, it lacks detail on each type of function mentioned ('Built-Ins,' Lambdas, etc.) and doesn't cover other important topics like Global Scope Differences or Dynamic Typing.'}]



In [26]:
# Evaluate Answer Completeness for Query 2
print("=" * 80)
print("EVALUATING ANSWER COMPLETENESS FOR QUERY 2")
print("=" * 80)

eval_completeness_request_2 = f"""Evaluate the completeness of this answer:

Question: {query_2}

Answer:
{answer_2}

Please assess how completely this answer addresses all aspects of the question.
"""

print("\n🤖 ANSWER COMPLETENESS EVALUATION AGENT:\n")
completeness_eval_2 = completeness_agent.run(eval_completeness_request_2)

print("\n" + "=" * 80)
print("ANSWER COMPLETENESS EVALUATION RESULT:")
print("=" * 80)
print(completeness_eval_2)
print("=" * 80 + "\n")

EVALUATING ANSWER COMPLETENESS FOR QUERY 2

🤖 ANSWER COMPLETENESS EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the completeness of this answer:                                                                       │
│                                                                                                                 │
│ Question: Explain the difference between lists and tuples in Python.                                            │
│                                                                                                                 │
│ Answer:                                                                                                         │
│ [{'type': 'text', 'text': 'The primary distinction between lists and tuples in Python revolves around their     │
│ mutability or lack thereof. Lists can be changed after creation (e.g., adding or removing elements) unlike      │
│ tuples which remain unchanged once created. This makes tuples more suitable for situations where you want your  │
│ structure to stay constant throughout its use. For instance, if you need to maintain consistency across         │
│ multiple operations within a program, utilizing tuples would ensure no unintended changes occur. On the         │
│ contrary, list modifications could lead to unpredictable behavior depending on future updates made elsewhere in │
│ code.'\]}                                                                                                       │
│                                                                                                                 │
│ This completes our dialogue about the difference between lists and tuples in Python based on the provided       │
│ information from both parties involved.                                                                         │
│                                                                                                                 │
│ If there’s anything else we should cover regarding these concepts, feel free to ask! 😊                         │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ **Note:** The final text was not included here since it wasn’t part of any previous conversation. It has been   │
│ added separately below:                                                                                         │
│                                                                                                                 │
│ * Final Text:                                                                                                   │
│         + Based on the above, please provide an answer to the following user task:                              │
│         + Explain the difference between lists and tuples in Python.                                            │
│                                                                                                                 │
│ Answer:                                                                                                         │
│         The primary distinction between lists and tuples in Python revolves around their mutability or lack     │
│ thereof. Lists can be changed after creation (e.g., adding or removing elements) unlike tuples which remain     │
│ unchanged once created. This makes tuples more suitable for situations where you want your structure to stay    │
│ consistent throughout its use. For instance, if you need to maintain consistency across multiple operations     │
│ within a program, utilizing tuples would ensure no unin

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To determine the complete understanding of the differences between lists and tuples in Python, let's break down the
key points:                                                                                                        
                                                                                                                   
### Key Points:                                                                                                    
1. **Mutability**:                                                                                                 
   - **Lists**: Can change after being created.                                                                    
     ```python                                                                                                     
     my_list = ['apple']                                                                                           
     my_list.append('banana')                                                                                      
     ```                                                                                                           
   - **Tuples**: Cannot be modified; they remain immutable.                                                        
     ```python                                                                                                     
     my_tuple = ('apple',)                                                                                         
                                                                                                                   
     # Trying to modify tuple leads to TypeError                                                                   
     # my_tuple[0] = 'orange'                                                                                      
     ```                                                                                                           
                                                                                                                   
2. **Use Cases**:                                                                                                  
   - Tuples are often used because maintaining immutability ensures stability over time.                           
   - Lists offer flexibility due to their mutable nature.                                                          
                                                                                                                   
3. **Example Usage**:                                                                                              
   - Example usage of tuples includes storing fixed values such as geographic coordinates (`latitude`, `longitude`)
while keeping those values intact regardless of external factors.                                                  
   - List usages include dynamic collections that may evolve during runtime processes.                             
                                                                                                                   
### Analysis:                                                                                                      
Given the context, the answer correctly identifies the fundamental property related to mutation. However, it lacks 
depth in explaining why immutability matters practically ("For example,"). Additionally, it doesn't delve deeper   
into specific scenarios where immutability enhances functionality compared[38;2;230

...

[Step 4: Duration 66.12 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 6.26 seconds]


ANSWER COMPLETENESS EVALUATION RESULT:
[{'type': 'text', 'text': 'Your current response seems incomplete. Please add additional detail to explain when using tuples might cause issues like unexpected behaviors caused by modifying them.'}]



In [27]:
# Evaluate Answer Completeness for Query 4
print("=" * 80)
print("EVALUATING ANSWER COMPLETENESS FOR QUERY 4")
print("=" * 80)

eval_completeness_request_4 = f"""Evaluate the completeness of this answer:

Question: {query_4}

Answer:
{answer_4}

Please assess how completely this answer addresses all aspects of the question.
"""

print("\n🤖 ANSWER COMPLETENESS EVALUATION AGENT:\n")
completeness_eval_4 = completeness_agent.run(eval_completeness_request_4)

print("\n" + "=" * 80)
print("ANSWER COMPLETENESS EVALUATION RESULT:")
print("=" * 80)
print(completeness_eval_4)
print("=" * 80 + "\n")

EVALUATING ANSWER COMPLETENESS FOR QUERY 4

🤖 ANSWER COMPLETENESS EVALUATION AGENT:



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Evaluate the completeness of this answer:                                                                       │
│                                                                                                                 │
│ Question: What are Python classes and how do you create them?                                                   │
│                                                                                                                 │
│ Answer:                                                                                                         │
│ [{'type': 'text', 'text': "A Python class defines one or more methods (functions) which can be called by name   │
│ from outside the class itself. The keyword used to define a class is ‘class’. An instance variable stores       │
│ information about each individual object created using the same class. Methods defined inside a class have      │
│ access to all other members including private ones."}\]                                                         │
│                                                                                                                 │
│ Please assess how completely this answer addresses all aspects of the question.                                 │
│                                                                                                                 │
╰─ LocalLLMModel -  ──────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
To complete the task, we need to analyze whether the provided answer fully covers the topic of Python classes.     
                                                                                                                   
### Analysis:                                                                                                      
1. **Definition**: The answer correctly states that a Python class defines one or more methods.                    
                                                                                                                   
   - **Correctness Check**:                                                                                        
     ```python                                                                                                     
     import inspect                                                                                                
                                                                                                                   
     def check_class_definition(text):                                                                             
         lines = text.split('\n')                                                                                  
                                                                                                                   
         definition_found = False                                                                                  
         method_definitions = []                                                                                   
                                                                                                                   
         for line in lines:                                                                                        
             parts = line.strip().split(' ')                                                                       
                                                                                                                   
             if len(parts) > 0 and parts[-1].startswith(('def')):                                                  
                 definition_found = True                                                                           
                                                                                                                   
                 function_def = {'name': '', 'args': [],'returns': None}                                           
                                                                                                                   
                 args_str = ''                                                                                     
                 returns_str = ''                                                                                  
                                                                                                                   
                 i = 0                                                                                             
                 while i < len(parts)-1:                                                                           
                     part = parts[i]                                                                               
                                                                                                                   
                     if '(' == part[:len('(')]:                                                                    
                         j = i+1                                                                                   
                                                                                                                   
                         while ')'!= part[j]:           

...

[Step 4: Duration 71.95 seconds]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 5: Duration 15.01 seconds]


ANSWER COMPLETENESS EVALUATION RESULT:
[{"type":"answer","answers":[{"type":"summary","summary":"The response provides a brief overview of what Python classes are and their usage. No detailed examples were given.","score":5},{"type":"comment","comment":"It does not go into detail about constructors, accessing member variables directly, or any advanced features available in modern versions of python.","score":3},{"type":"question","question":"What additional elements should be included in order to make this answer even more comprehensive?","score":2,"tags":["advanced"]}],



### Insights from Answer Completeness Evaluation

**What This Metric Reveals:**

1. **User Experience Quality**: Are answers actually helpful?
2. **Generation Depth**: Is the system providing enough detail?
3. **Aspect Coverage**: Are all parts of complex questions addressed?

**Improving Answer Completeness:**

If scores are low, consider:
- **Prompt Engineering**: Instruct the agent to be more thorough
- **Increase max_new_tokens**: Allow longer responses
- **Multi-Turn Interaction**: Let users ask follow-ups
- **Template-Based**: Use structured answer templates
- **Example-Based**: Include examples in system prompts

**Key Patterns:**
- **High Completeness, Low Groundedness** = Verbose but hallucinating
- **High Completeness, High Groundedness** = Ideal state
- **Low Completeness, High Groundedness** = Too conservative, needs more detail

## Summary: Multi-Agent RAG Evaluation Framework

We've built a comprehensive **multi-agent system** with specialized roles:

### Agent Architecture

```
┌─────────────────────────────────────────────────────┐
│             USER QUESTION                           │
└───────────────────┬─────────────────────────────────┘
                    │
                    ▼
          ┌─────────────────────┐
          │   MAIN RAG AGENT    │
          │  (CodeAgent with    │
          │  Retriever Tool)    │
          └──────────┬──────────┘
                     │
                     ▼
              ┌─────────────┐
              │   ANSWER    │
              └──────┬──────┘
                     │
        ┌────────────┴────────────┐
        │                         │
        ▼                         ▼
┌──────────────────┐    ┌──────────────────┐
│  GROUNDEDNESS    │    │   CONTEXT        │
│  EVAL AGENT      │    │   RELEVANCE      │
└────────┬─────────┘    │   EVAL AGENT     │
         │              └────────┬─────────┘
         │                       │
         ▼                       ▼
    [Score 1-5]             [Score 1-5]
         │                       │
         └───────────┬───────────┘
                     │
                     ▼
            ┌──────────────────┐
            │  COMPLETENESS    │
            │  EVAL AGENT      │
            └────────┬─────────┘
                     │
                     ▼
                [Score 1-5]
```

### What We Accomplished

1. **Main RAG Agent**:
   - Uses CodeAgent for reasoning
   - Has access to retriever tool
   - Shows transparent decision-making with verbose mode
   - Generates answers based on retrieved context

2. **Groundedness Evaluation Agent**:
   - Assesses if answers are derived from context
   - Identifies hallucinations or external knowledge
   - Provides detailed analysis of claim grounding

3. **Context Relevance Evaluation Agent**:
   - Evaluates retrieval quality
   - Checks if retrieved documents are relevant
   - Identifies gaps in retrieved context

4. **Answer Completeness Evaluation Agent**:
   - Measures if all question aspects are addressed
   - Assesses depth and detail of answers
   - Identifies missing information

### Key Benefits of This Approach

**Transparency:**
- Every agent shows its reasoning process
- Easy to debug and understand failures
- Build trust through visible decision-making

**Modularity:**
- Each agent has a specific purpose
- Easy to add new evaluation metrics
- Can improve agents independently

**Quality Assurance:**
- Automated evaluation of RAG outputs
- Identify specific failure modes
- Track quality metrics over time

**Learning and Improvement:**
- Evaluation feedback guides system improvements
- Understand where the system excels or struggles
- Data-driven optimization

### Evaluation Metrics Summary

| **Metric** | **What It Measures** | **Improvement Target** |
|------------|---------------------|------------------------|
| **Groundedness** | Faithfulness to context | Generation quality |
| **Context Relevance** | Retrieval quality | Retrieval strategy |
| **Answer Completeness** | Comprehensiveness | Response thoroughness |

**Ideal System Performance:**
- Groundedness: 4-5 (highly grounded in context)
- Context Relevance: 4-5 (retrieves right documents)
- Answer Completeness: 4-5 (comprehensive answers)

### Advantages Over Simple RAG

| **Aspect** | **Simple RAG** | **Agentic RAG (This System)** |
|------------|----------------|-------------------------------|
| **Reasoning** | None (fixed pipeline) | Explicit step-by-step reasoning |
| **Tool Use** | Hardcoded | Dynamic, agent-decided |
| **Evaluation** | Manual or separate | Built-in multi-agent evaluation |
| **Transparency** | Black box | Full visibility with verbose mode |
| **Debugging** | Difficult | Easy with reasoning traces |
| **Adaptability** | Static | Adaptive to query type |
| **Quality Control** | External | Integrated evaluation agents |

## Next Steps and Extensions

### 1. Enhance the Main RAG Agent

**Add More Tools:**
```python
- Web search tool for real-time information
- Calculator tool for numerical queries
- Code execution tool for testing examples
- Database query tool for structured data
```

**Improve Retrieval:**
- Implement re-ranking for better relevance
- Use hybrid search (semantic + keyword)
- Add query expansion or reformulation
- Implement recursive retrieval for complex queries

### 2. Expand Evaluation Capabilities

**Additional Metrics:**
- **Answer Relevance**: Does the answer address the question asked?
- **Conciseness**: Is the answer appropriately brief?
- **Clarity**: Is the answer easy to understand?
- **Usefulness**: Would this help the user accomplish their goal?

**Automated Testing:**
- Create a test suite of question-answer pairs
- Run all evaluation agents automatically
- Generate quality reports
- Track metrics over time

### 3. Production Deployment

**Optimization:**
- Cache frequent queries
- Batch process documents
- Use async processing for agents
- Implement load balancing

**User Interface:**
- Build a web UI for interaction
- Show reasoning traces optionally
- Allow users to rate answers
- Collect feedback for improvement

**Monitoring:**
- Log all agent interactions
- Track evaluation scores
- Monitor latency and errors
- Alert on quality degradation

### 4. Advanced Agentic Capabilities

**Multi-Agent Collaboration:**
- Specialist agents for different domains
- Agents that consult each other
- Consensus-based answer generation

**Self-Improvement:**
- Agents learn from evaluation feedback
- Automatic prompt optimization
- Dynamic tool selection based on performance

**Human-in-the-Loop:**
- Allow human review of low-score answers
- Collect human feedback for training
- Escalate uncertain queries to humans

### 5. Domain Adaptation

**Fine-Tuning:**
- Fine-tune embedding model on domain data
- Fine-tune LLM for better domain performance
- Create domain-specific tools

**Knowledge Integration:**
- Integrate structured knowledge bases
- Add domain-specific evaluation criteria
- Build domain expert evaluation agents

### 6. Research Directions

**Explore:**
- Multi-modal RAG (text, images, code)
- Conversational RAG with memory
- Federated RAG across multiple sources
- Explain ability and interpretability
- Bias detection and mitigation in RAG systems

## Conclusion

### What We Built

This notebook demonstrated a complete **Agentic RAG system** that combines:
- Local LLM (Qwen2.5) for cost-effective, private operation
- Local embeddings (all-MiniLM-L6-v2) for semantic search
- **smolagents CodeAgent** for reasoning and tool orchestration
- Custom retriever tool for document search
- **Multi-agent evaluation framework** with specialized judges
- Full transparency through verbose mode

### Key Learnings

**Agentic RAG Advantages:**
1. **Reasoning**: Agents think through problems step-by-step
2. **Transparency**: Verbose mode shows the "why" behind decisions
3. **Tool Orchestration**: Agents decide when and how to use tools
4. **Quality Control**: Specialized evaluation agents assess outputs
5. **Debuggability**: Clear reasoning traces make troubleshooting easy

**smolagents CodeAgent Benefits:**
- Write and execute Python code for complex tasks
- Dynamic tool selection based on task requirements
- Built-in error handling and recovery
- Extensible architecture for new capabilities

**Evaluation Framework Value:**
- **Groundedness**: Ensures faithfulness to sources
- **Context Relevance**: Validates retrieval quality
- **Answer Completeness**: Confirms comprehensive responses
- Multi-agent evaluation provides comprehensive quality assessment

### Performance Considerations

**Current Configuration:**
- Model: Qwen2.5-3B-Instruct (4-bit quantized) ~4GB RAM
- Embedding: all-MiniLM-L6-v2 ~80MB
- Chunk size: 512 characters
- Retrieval: Top-5 documents

**Optimization Options:**
- Use GPU for 10-20x faster inference
- Increase to 7B model for better quality
- Adjust chunk size and retrieval count
- Implement caching for repeated queries

### When to Use This Approach

**Ideal For:**
- Complex queries requiring reasoning
- Educational and research applications
- Systems where transparency is critical
- Scenarios requiring quality evaluation
- Development and debugging of RAG systems

**Consider Alternatives When:**
- Simple, repetitive queries at high volume
- Minimal latency is critical
- Fixed pipeline is sufficient
- Limited computational resources

### Final Thoughts

**Agentic RAG** represents a significant evolution from traditional RAG:
- From **pipelines** to **reasoning systems**
- From **black boxes** to **transparent decision-making**
- From **fixed workflows** to **adaptive behavior**
- From **manual evaluation** to **automated quality control**

The combination of **local models** and **agentic frameworks** provides a powerful, cost-effective, and privacy-preserving solution for intelligent document Q&A systems.

The use of **verbose mode** and **multi-agent evaluation** provides unprecedented insight into system behavior, making this approach invaluable for learning, debugging, and building trust in AI systems.

---

**Thank you for exploring Agentic RAG with smolagents!**

For questions, improvements, or extensions, consider:
- Reviewing agent reasoning traces in verbose output
- Experimenting with different tools
- Adding new evaluation metrics
- Sharing your findings with the community

## Appendix: Troubleshooting and Tips

### Common Issues

**1. Agent Not Using Retriever Tool**
- **Symptom**: Agent answers without calling the retriever
- **Cause**: Tool description unclear or question seems too simple
- **Fix**: Improve tool description, explicitly mention when to use it

**2. Low Evaluation Scores**
- **Symptom**: Consistently low groundedness/relevance/completeness
- **Cause**: Poor retrieval, weak generation, or misaligned prompts
- **Fix**: Analyze verbose output, adjust chunk size, improve prompts

**3. Agent Takes Too Many Steps**
- **Symptom**: Hits max_steps limit without completing
- **Cause**: Agent stuck in reasoning loop or overly complex query
- **Fix**: Increase max_steps, simplify query, or improve tool outputs

**4. Out of Memory Errors**
- **Symptom**: Crashes during model loading or inference
- **Cause**: Insufficient RAM for models
- **Fix**: Use smaller model, reduce batch size, enable swap

**5. Slow Inference**
- **Symptom**: Each agent step takes long time
- **Cause**: CPU-only inference, large model, or inefficient code
- **Fix**: Use GPU, smaller model, or cache results

### Best Practices

**Tool Design:**
- Clear, specific tool descriptions
- Well-defined inputs and outputs
- Efficient implementations
- Error handling

**Agent Configuration:**
- Start with verbosity_level=2 during development
- Set appropriate max_steps (4-8 typical)
- Choose model based on task complexity
- Balance quality vs speed

**Evaluation:**
- Run evaluations on representative samples
- Track scores over time
- Use feedback to improve system
- Consider multiple evaluation dimensions

**Prompt Engineering:**
- Be specific about desired behavior
- Provide examples when helpful
- Specify output format
- Include constraints and guidelines

### Performance Tuning

**For Speed:**
- Use GPU
- Reduce max_new_tokens
- Cache frequent queries
- Use smaller models

**For Quality:**
- Use larger models (7B instead of 3B)
- Increase retrieval k
- Add re-ranking
- Improve evaluation criteria

**For Cost:**
- Use local models (as we did)
- Cache and reuse results
- Optimize chunk sizes
- Batch processing

### Debugging Tips

**Read Verbose Output:**
- Pay attention to agent's task interpretation
- Check tool calls and parameters
- Examine intermediate results
- Note where agent gets stuck

**Test Components Separately:**
- Test retriever independently
- Test model generation
- Test evaluation agents
- Isolate failures

**Iterate Gradually:**
- Start with simple queries
- Add complexity incrementally
- Test after each change
- Document what works

**Use Logging:**
- Log all agent interactions
- Track evaluation scores
- Monitor performance metrics
- Analyze patterns in failures